# Notebook 8 — Model Export & FastAPI Integration

**Goal:** Export a single generalised production pipeline for cross-domain RUL inference
using CNN-LSTM-DANN weights and `FeatureAligner`, then verify end-to-end behavior.

**Pipeline contents:**
1. Savitzky-Golay smoothing
2. Feature alignment (any sensor count → 24 features)
3. Time-window construction
4. CNN-LSTM model inference (DANN-adapted weights)
5. CUSUM health-state classifier


In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import joblib
from scipy.signal import savgol_filter
from sklearn.base import BaseEstimator, RegressorMixin

from src.feature_aligner      import FeatureAligner
from src.models.cnn_lstm      import build_cnn_lstm
from src.models.cnn_lstm_dann import build_cnn_lstm_dann
from src.changepoint          import cusum_detector, classify_health_state

WINDOW_SIZE = 30
MAX_RUL     = 125


## 8.1 Define the Production Pipeline Wrapper


In [8]:
class GeneralisedMaintenancePipeline(BaseEstimator, RegressorMixin):
    """
    Production pipeline for factory machinery RUL prediction.

    Accepts any number of sensor columns via FeatureAligner.
    Uses CNN-LSTM-DANN weights adapted from FD001 to AI4I.
    """

    def __init__(self, aligner, model_weights_path, target_dim=24,
                  window_size=30, max_rul=125, cusum_threshold=4.5,
                  sg_window=11, sg_poly=3):
        self.aligner             = aligner
        self.model_weights_path  = model_weights_path
        self.target_dim          = target_dim
        self.window_size         = window_size
        self.max_rul             = max_rul
        self.cusum_threshold     = cusum_threshold
        self.sg_window           = sg_window
        self.sg_poly             = sg_poly
        self._model              = None

    def _load_model(self):
        if self._model is None:
            reg_model, adv_model = build_cnn_lstm_dann(
                window_size=self.window_size,
                n_features=self.target_dim
            )
            adv_model.load_weights(self.model_weights_path)
            self._model = reg_model

    def predict(self, X_raw: np.ndarray) -> dict:
        self._load_model()
        X = X_raw.astype(np.float64).copy()

        if len(X) >= self.sg_window:
            for j in range(X.shape[1]):
                X[:, j] = savgol_filter(X[:, j], self.sg_window, self.sg_poly)

        expected_in = getattr(self.aligner, 'input_dim', X.shape[1])
        if X.shape[1] < expected_in:
            pad = np.zeros((X.shape[0], expected_in - X.shape[1]))
            X = np.hstack([X, pad])
        elif X.shape[1] > expected_in:
            X = X[:, :expected_in]

        X_aligned = self.aligner.transform(X)

        T = len(X_aligned)
        if T < self.window_size:
            pad       = np.zeros((self.window_size - T, X_aligned.shape[1]))
            X_aligned = np.vstack([pad, X_aligned])
        window = X_aligned[-self.window_size:][np.newaxis].astype(np.float32)

        rul = float(np.clip(
            self._model.predict(window, verbose=0).flatten()[0] * self.max_rul,
            0, self.max_rul
        ))

        n_r = min(50, len(X_aligned))
        cp  = cusum_detector(X_aligned[-n_r:, 0], threshold=self.cusum_threshold)
        health = classify_health_state(rul, cp is not None)

        return {
            'rul_prediction':        round(rul, 1),
            'health_state':          health,
            'change_point_detected': cp is not None,
            'change_point_step':     int(cp) if cp is not None else None,
            'n_input_sensors':       X_raw.shape[1],
            'alignment_method':      self.aligner.summary()['method']
        }

    def __getstate__(self):
        state = self.__dict__.copy(); state['_model'] = None; return state

    def __setstate__(self, state):
        self.__dict__.update(state); self._model = None


## 8.2 Export Generalised Pipeline


In [11]:
os.makedirs('../models/saved', exist_ok=True)

aligner      = FeatureAligner.load('../models/saved/aligner_ai4i.joblib')
dann_weights = '../models/saved/cnn_lstm_dann_FD001_to_AI4I.weights.h5'

if not os.path.exists(dann_weights):
    print('❌ DANN weights not found. Run Notebook 05 first.')
else:
    pipeline = GeneralisedMaintenancePipeline(
        aligner            = aligner,
        model_weights_path = os.path.abspath(dann_weights),
        target_dim         = 24,
        window_size        = WINDOW_SIZE,
        max_rul            = MAX_RUL,
        cusum_threshold    = 4.5
    )
    pipeline_path = '../models/saved/pm_pipeline_generalised.joblib'
    joblib.dump(pipeline, pipeline_path)
    print(f'✅ Generalised pipeline saved: {pipeline_path}')
    print('   Accepts: any number of sensor columns')
    print('   Trained on: FD001 (turbofan) → adapted to AI4I (factory)')


✅ Generalised pipeline saved: ../models/saved/pm_pipeline_generalised.joblib
   Accepts: any number of sensor columns
   Trained on: FD001 (turbofan) → adapted to AI4I (factory)


## 8.3 Test the Generalised Pipeline


In [12]:
pipeline = joblib.load('../models/saved/pm_pipeline_generalised.joblib')

ai4i_raw = pd.read_csv('../data/raw/ai4i2020.csv')
AI4I_SENSOR_COLS = ['Air temperature [K]', 'Process temperature [K]',
                    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

sample_X = ai4i_raw[AI4I_SENSOR_COLS].values[:100]
result   = pipeline.predict(sample_X)

print('Pipeline test on AI4I data (5 sensors):')
print(f"  Input sensors:  {result['n_input_sensors']}")
print(f"  Alignment:      {result['alignment_method']}")
print(f"  Predicted RUL:  {result['rul_prediction']} cycles")
print(f"  Health State:   {result['health_state']}")
print(f"  Change Point:   {result['change_point_detected']}")

sample_3sensor = np.random.rand(80, 3)
result_3s = pipeline.predict(sample_3sensor)
print('\nPipeline test on hypothetical 3-sensor machine:')
print(f"  Input sensors:  {result_3s['n_input_sensors']}")
print(f"  Alignment:      {result_3s['alignment_method']}")
print(f"  Predicted RUL:  {result_3s['rul_prediction']} cycles")


Pipeline test on AI4I data (5 sensors):
  Input sensors:  5
  Alignment:      zero_pad
  Predicted RUL:  62.9 cycles
  Health State:   Warning
  Change Point:   True

Pipeline test on hypothetical 3-sensor machine:
  Input sensors:  3
  Alignment:      zero_pad
  Predicted RUL:  20.9 cycles


In [ ]:
# Verify prediction changes with engine age
df_train_fd001 = datasets['FD001']['train']
unit_max_cycle = df_train_fd001[df_train_fd001['unit_id'] == 1]['cycle'].max()

rul_trajectory = []
for pct in [0.2, 0.4, 0.6, 0.8, 1.0]:
    n_cycles = int(unit_max_cycle * pct)
    raw_data = df_train_fd001[
        (df_train_fd001['unit_id'] == 1) &
        (df_train_fd001['cycle'] <= n_cycles)
    ][FEATURE_COLS].values
    r = pipeline.predict(raw_data)
    rul_trajectory.append({
        'Pct Lifetime': f'{pct*100:.0f}%',
        'Cycles Used':  n_cycles,
        'Predicted RUL': r['rul_prediction'],
        'Health State':  r['health_state'],
        'Change Point':  r['change_point_detected']
    })

traj_df = pd.DataFrame(rul_trajectory)
print("\nPipeline predictions at different lifecycle stages:")
print(traj_df.to_string(index=False))


In [ ]:
# Visualise predicted RUL over lifecycle stages
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(traj_df['Cycles Used'], traj_df['Predicted RUL'],
        marker='o', linewidth=2.5, color='steelblue', markersize=8)
ax.axhline(50, color='orange', linestyle='--', linewidth=1.5, label='Warning threshold (50)')
ax.axhline(20, color='red',    linestyle='--', linewidth=1.5, label='Critical threshold (20)')

for _, row in traj_df.iterrows():
    color = ('red' if row['Health State'] == 'Critical'
             else ('orange' if row['Health State'] == 'Warning' else 'green'))
    ax.annotate(row['Health State'],
                xy=(row['Cycles Used'], row['Predicted RUL']),
                xytext=(0, 12), textcoords='offset points',
                ha='center', color=color, fontsize=9)

ax.set_xlabel('Cycles of Data Available')
ax.set_ylabel('Predicted RUL')
ax.set_title('Pipeline Predictions at Different Lifecycle Stages (Engine 1, FD001)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Insight:** The predicted RUL decreases monotonically as more cycles become
available and the engine approaches failure. The health state transitions from
Healthy → Warning → Critical, providing a natural three-stage alert system
for factory operators.


## 8.4 FastAPI Startup Check


In [ ]:
# Verify API can start by checking import paths
print("FastAPI module check:")
try:
    from api.schemas   import PredictRequest, PredictResponse
    from api.predictor import run_prediction, list_available_models
    from api.main      import app
    print("  All API modules imported successfully.")
    models_available = list_available_models(models_dir='../models/saved')
    print(f"  Available models: {models_available}")
except ImportError as e:
    print(f"  Import error: {e}")
    print("  Ensure api/__init__.py exists and all api/ files are created.")


## 8.5 API Startup & Test Instructions


In [ ]:
startup_instructions = """
To launch the FastAPI server, run from the project root directory:

    uvicorn api.main:app --host 0.0.0.0 --port 8000 --reload

Then open your browser to:
    http://localhost:8000/docs   ← Swagger UI (interactive API explorer)
    http://localhost:8000/redoc  ← ReDoc documentation

Available endpoints:
    GET  /health   → liveness check
    GET  /models   → list available dataset models
    POST /predict  → RUL prediction + health state + explanation

Example Python request (run after starting the server):

    import requests, json
    payload = {
        "unit_id":    "engine_001",
        "dataset_id": "FD001",
        "readings":   raw_sensor_data.tolist()   # shape: (n_cycles, 24)
    }
    r = requests.post("http://localhost:8000/predict", json=payload)
    print(json.dumps(r.json(), indent=2))
"""
print(startup_instructions)


In [ ]:
# Live API test (run this cell AFTER starting the server in a separate terminal)
try:
    # Test /health
    r = requests.get("http://localhost:8000/health", timeout=3)
    print("Health check:", r.json())

    # Test /models
    r = requests.get("http://localhost:8000/models", timeout=3)
    print("Available models:", r.json())

    # Test /predict
    unit_1_readings = df_test[df_test['unit_id'] == 1][FEATURE_COLS].values
    payload = {
        "unit_id":    "engine_001",
        "dataset_id": "FD001",
        "readings":   unit_1_readings.tolist()
    }
    r = requests.post("http://localhost:8000/predict", json=payload, timeout=10)
    print("\nPrediction response:")
    print(json.dumps(r.json(), indent=2))

except requests.exceptions.ConnectionError:
    print("Server not running. Start it with: uvicorn api.main:app --port 8000")


## 8.6 Final Export Summary


In [ ]:
print("=" * 55)
print("EXPORTED FILES IN models/saved/")
print("=" * 55)
for f in sorted(os.listdir('../models/saved')):
    size_kb = os.path.getsize(f'../models/saved/{f}') / 1024
    print(f"  {f:<45} {size_kb:>8.1f} KB")

print("\n" + "=" * 55)
print("DEPLOYMENT CHECKLIST")
print("=" * 55)
checklist = [
    "pm_pipeline_fd001.joblib  — FD001 full inference pipeline",
    "pm_pipeline_fd002.joblib  — FD002 full inference pipeline",
    "pm_pipeline_fd003.joblib  — FD003 full inference pipeline",
    "pm_pipeline_fd004.joblib  — FD004 full inference pipeline",
    "lstm_target_only_*.keras  — LSTM weights per dataset",
    "scaler_*.joblib           — MinMaxScaler per dataset",
]
for item in checklist:
    path_key = item.split('—')[0].strip().replace('*', 'FD001')
    exists = os.path.exists(f'../models/saved/{path_key}')
    status = "✓" if exists else "✗ (MISSING)"
    print(f"  [{status}] {item}")

print("\nTo add a NEW machine type:")
print("  1. Collect run-to-failure sensor data in (n_cycles × 24) format")
print("  2. Run full_preprocess_pipeline() → save scaler as scaler_NEWTYPE.joblib")
print("  3. Train LSTM → save weights as lstm_target_only_NEWTYPE.keras")
print("  4. Create PredictiveMaintenancePipeline → save as pm_pipeline_newtype.joblib")
print("  5. Call POST /predict with dataset_id='NEWTYPE'")
